In [ ]:
%pip install scikit-learn #Usa %pip install em vez de apenas pip install porque o primeiro instala os pacotes diretamente no ambiente do kernel.

In [22]:
import sklearn
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

from sklearn.model_selection import train_test_split
from time import perf_counter
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [36]:
PASTA_PRECIP = os.path.join("..", "DATA", "CSV_regioes")   # precipitação
PASTA_ACUDES = os.path.join("..", "DATA", "CSV_acudes")    # volume dos açudes

def ler_pasta(pasta, sep=";", **kwargs):
    lista = []
    for nome in sorted(os.listdir(pasta)):
        if nome.lower().endswith(".csv"):
            df = pd.read_csv(os.path.join(pasta, nome), sep=sep, **kwargs)
            df["arquivo"] = os.path.splitext(nome)[0]
            lista.append(df)
    if not lista:
        raise FileNotFoundError(f"Nenhum CSV em: {pasta}")
    return lista

df_1 = pd.concat(ler_pasta(PASTA_PRECIP, decimal="."), ignore_index=True)
df_2 = pd.concat(ler_pasta(PASTA_ACUDES, decimal=","), ignore_index=True).rename(columns={"arquivo": "acude"})

display(df_1.head())
display(df_2.head())
print(df_1.dtypes, "\n")
print(df_2.dtypes)

,data,lat,lon,precip_mm,municipio,estado,uf,regiao,arquivo
0,2006-01-01,-7.75,-41.25,5.292850,Jacobina do Piauí,Piauí,PI,Nordeste,precipitacao_2006_com_local
1,2006-01-01,-7.75,-40.75,10.205985,Simões,Piauí,PI,Nordeste,precipitacao_2006_com_local
2,2006-01-01,-7.75,-40.25,20.805733,Trindade,Pernambuco,PE,Nordeste,precipitacao_2006_com_local
3,2006-01-01,-7.75,-39.75,6.968030,Granito,Pernambuco,PE,Nordeste,precipitacao_2006_com_local
4,2006-01-01,-7.75,-39.25,0.812125,Cedro,Pernambuco,PE,Nordeste,precipitacao_2006_com_local


,DateTime,Volume (hm³),acude
0,2006-01-01 00:00:00,42.59,reserv_atalho
1,2006-01-02 00:00:00,42.59,reserv_atalho
2,2006-01-03 00:00:00,NaN,reserv_atalho
3,2006-01-04 00:00:00,NaN,reserv_atalho
4,2006-01-05 00:00:00,NaN,reserv_atalho


data             str
lat          float64
lon          float64
precip_mm    float64
municipio        str
estado           str
uf               str
regiao           str
arquivo          str
dtype: object 

DateTime            str
Volume (hm³)    float64
acude               str
dtype: object


In [37]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Distância em km entre pontos (fórmula de Haversine)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

coordenadas_acudes = pd.DataFrame({
    "acude":     ["reserv_atalho", "reserv_rosario", "reserv_olhodgua", "reserv_quixabinha"],
    "lat_acude": [-8.75, -6.75, -6.25, -7.25],
    "lon_acude": [-38.25, -39.25, -36.25, -38.75],
})
RAIO_MAXIMO_KM = 50.0

# chave de data em texto, igual nas duas tabelas
df_1["data_str"] = pd.to_datetime(df_1["data"]).dt.strftime("%Y-%m-%d")
df_2["data_str"] = pd.to_datetime(df_2["DateTime"]).dt.strftime("%Y-%m-%d")

# remove linhas repetidas de precipitação (mesma data e mesmo ponto)
antes = len(df_1)
df_1 = df_1.drop_duplicates(subset=["data_str", "lat", "lon"]).reset_index(drop=True)
print(f"Linhas repetidas removidas em df_1: {antes - len(df_1)}")

pontos = df_1[["lat", "lon"]].drop_duplicates().reset_index(drop=True)
mapeamento = []
for _, ac in coordenadas_acudes.iterrows():
    dist = haversine_km(ac["lat_acude"], ac["lon_acude"], pontos["lat"], pontos["lon"])
    i = dist.idxmin()
    if dist.loc[i] <= RAIO_MAXIMO_KM:
        mapeamento.append({
            "acude": ac["acude"],
            "lat_precip_mais_proxima": pontos.loc[i, "lat"],
            "lon_precip_mais_proxima": pontos.loc[i, "lon"],
            "distancia_km": dist.loc[i],
        })
    else:
        print(f"Açude '{ac['acude']}' ignorado: ponto de chuva mais próximo a {dist.loc[i]:.1f} km (limite {RAIO_MAXIMO_KM} km).")

df_map = pd.DataFrame(mapeamento)
if df_map.empty:
    raise ValueError("Nenhum açude atendeu ao critério de distância.")

df_2_geo = df_2.merge(df_map, on="acude", how="inner")
df_unificado = df_2_geo.merge(
    df_1,
    left_on=["data_str", "lat_precip_mais_proxima", "lon_precip_mais_proxima"],
    right_on=["data_str", "lat", "lon"],
    how="inner",
)
print(f"Registros válidos (raio de {RAIO_MAXIMO_KM} km): {len(df_unificado)}")
print(df_unificado["acude"].value_counts())

df_final = df_unificado.rename(columns={
    "DateTime": "Data",
    "Volume (hm³)": "volume",
    "precip_mm": "precipitação",
    "classe_acude": "classe_volume_acude",
    "status_alerta": "bin_status_alerta",
})
cols = ["Data", "acude", "municipio", "estado", "volume", "pct",
        "precipitação", "classe_volume_acude", "bin_status_alerta"]
df_final = df_final[[c for c in cols if c in df_final.columns]].copy()

print(df_final.dtypes)
print(df_final.isna().sum())
display(df_final.head())

Linhas repetidas removidas em df_1: 198
Açude 'reserv_atalho' ignorado: ponto de chuva mais próximo a 111.2 km (limite 50.0 km).
Açude 'reserv_olhodgua' ignorado: ponto de chuva mais próximo a 110.5 km (limite 50.0 km).
Registros válidos (raio de 50.0 km): 15154
acude
reserv_quixabinha    7577
reserv_rosario       7577
Name: count, dtype: int64
Data                str
acude               str
municipio           str
estado              str
volume          float64
precipitação    float64
dtype: object
Data               0
acude              0
municipio          0
estado             0
volume          1417
precipitação       2
dtype: int64


,Data,acude,municipio,estado,volume,precipitação
0,2006-01-01 00:00:00,reserv_quixabinha,Mauriti,Ceará,7.58,1.460885
1,2006-01-02 00:00:00,reserv_quixabinha,Mauriti,Ceará,7.58,44.768280
2,2006-01-03 00:00:00,reserv_quixabinha,Mauriti,Ceará,7.57,18.526501
3,2006-01-04 00:00:00,reserv_quixabinha,Mauriti,Ceará,7.57,0.000000
4,2006-01-05 00:00:00,reserv_quixabinha,Mauriti,Ceará,7.57,0.000000


In [79]:
# 1. Reconstruir df_final a partir de df_unificado usando 'data_str' (sem risco de duplicar 'Data')
df_final = df_unificado.rename(columns={
    "data_str": "Data",
    "Volume (hm³)": "Volume",
    "precip_mm": "precipitação",
}).copy()

# Remove eventuais colunas com nomes duplicados, mantendo a primeira ocorrência
df_final = df_final.loc[:, ~df_final.columns.duplicated()].copy()

# 2. Converter Data e ordenar por açude e data
df_final["Data"] = pd.to_datetime(df_final["Data"])
df_final = df_final.sort_values(["acude", "Data"]).reset_index(drop=True)

# 3. Criar features defasadas (Lag features) por açude
df_final["Volume Anterior"] = df_final.groupby("acude")["Volume"].shift(1)
df_final["Precipitacao_Anterior"] = df_final.groupby("acude")["precipitação"].shift(1)

# 4. Renomear precipitação atual
df_final = df_final.rename(columns={"precipitação": "Precipitacao"})

# 5. Selecionar colunas e remover NaNs
colunas_selecionadas = [
    "acude",
    "Data",
    "Volume",
    "Volume Anterior",
    "Precipitacao",
    "Precipitacao_Anterior",
]

df_reduzido = df_final[colunas_selecionadas].dropna().reset_index(drop=True)

# 6. Transformar 'acude' em variável quantitativa (0 = Quixabinha, 1 = Rosário)
df_reduzido = pd.get_dummies(df_reduzido, columns=["acude"], drop_first=True, dtype=int)

# 7. Formatar a data para dia/mês/ano
df_reduzido["Data"] = df_reduzido["Data"].dt.strftime("%d/%m/%Y")

display(df_reduzido.head())

,Data,Volume,Volume Anterior,Precipitacao,Precipitacao_Anterior,acude_reserv_rosario
0,02/01/2006,7.58,7.58,44.768280,1.460885,0
1,03/01/2006,7.57,7.58,18.526501,44.768280,0
2,04/01/2006,7.57,7.57,0.000000,18.526501,0
3,05/01/2006,7.57,7.57,0.000000,0.000000,0
4,06/01/2006,7.57,7.57,0.000000,0.000000,0


- Teste de Regressão Supervisionada

Simples Linear Regression com mais que um preditor

In [ ]:
# 1.1 Organiza os dados em matriz de features e vetor alvo
y2 = df_reduzido['Volume']  # variável alvo (o que queremos prever): Volume
X2 = df_reduzido[['Volume Anterior', 'Precipitacao']]  # variáveis preditoras: 2 colunas numéricas

'''
 Divide os dados entre treino e teste
 test_size=.2 -> 20% dos dados vão para teste e 80% para treino
 random_state=1 -> fixa a "semente" aleatória, para a divisão ser sempre a mesma (reprodutível)
'''
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=.2, random_state=1)

# Cria o modelo de regressão linear
# fit_intercept=True -> o modelo calcula o intercepto (b), ou seja, o valor de y quando X = 0
LM2 = LinearRegression(fit_intercept=True)

# Treina o modelo: encontra os coeficientes que melhor ajustam X2_train a y2_train
LM2.fit(X2_train, y2_train)


print('model.coef_ (w):', LM2.coef_) # Mostra os coeficientes (pesos) aprendidos, um para cada variável preditora
print('model.intercept_ (b):', LM2.intercept_) # Mostra o intercepto (b) da equação

print(f'f(X) = {LM2.intercept_:.2f} +\
 {LM2.coef_[0]:.2f}*{X2_train.columns[0]}  +\
 {LM2.coef_[1]:.2f}*{X2_train.columns[1]}')

# Calcula e mostra o R² no conjunto de treino
# (coeficiente de determinação: proporção da variação de y explicada pelo modelo, de 0 a 1)
print(f'LM2.score (R^2) in the train set: {LM2.score(X2_train, y2_train):.2f}')

# Usa o modelo treinado para prever o Volume nos dados de teste (que ele nunca viu)
y_test_pred = LM2.predict(X2_test)

model.coef_ (w): [9.99913340e-01 8.62245141e-04]
model.intercept_ (b): -0.016684458629597998
f(X) = -0.02 + 1.00*Volume Anterior  + 0.00*Precipitacao
LM2.score (R^2) in the train set: 1.00


O modelo está basicamente copiando o volume anterior..

Volume Anterior: ≈ 1,00
Para cada 1 unidade a mais no volume anterior, o volume atual sobe ~1 unidade. Ou seja, o modelo diz que Volume de hoje ≈ Volume de ontem.

Precipitação: ≈ 0,00
O coeficiente é minúsculo. Cada 1 mm a mais de chuva soma só ~0,0009 ao volume, o que é praticamente nada. Atenção: isso depende da escala.

Intercepto: ≈ -0,02
Praticamente zero. Se o volume anterior e a precipitação fossem 0, o modelo preveria ~0. Faz sentido físico.

1.  A precipitação quase não contribui nesse modelo, porque o Volume Anterior já “carrega” quase toda a informação.
2. O R² alto engana. Um modelo ingênuo (“amanhã = hoje”) teria desempenho parecido.

In [95]:
models_label = [] # to accumulate models labels
models_R2_train = [] # to accumulate models R^2 in the train phase
models_R2_test = [] # to accumulate models R^2 in the test phase
def computeScores(models, X_train=X_train, X_test=X_test,
                  y_train=y_train, y_test=y_test):
    for model in models:
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        formalism = type(model).__name__
        models_label.append(formalism)
        models_R2_train.append(model.score(X_train, y_train))
        models_R2_test.append(model.score(X_test, y_test))
        # print(formalism, " ok ")
    df = pd.DataFrame({'model': models_label,
                       'R2_train': models_R2_train,
                       'R2_test': models_R2_test})
    print(" *********** Rank based on R^2 in the test set *********** ")
    display(df.sort_values(by="R2_test", ascending = False))


In [ ]:
computeScores([LM2], X_train=X2_train, X_test=X2_test, y_train=y2_train, y_test=y2_test)

 *********** Rank based on R^2 in the test set *********** 


,model,R2_train,R2_test
0,LinearRegression,0.999833,0.999889
